# Product Detail Parser

Parses saved product-detail HTML into structured fields: Best Sellers Rank (main category and sub-category), 'bought in past month' badge, specs (including unit count) and ingredients.

**Setup.** Data paths point to `/content/drive/MyDrive/YOUR_PROJECT_FOLDER/...` — change `YOUR_PROJECT_FOLDER` to your own folder before running. API keys are read from Colab Secrets / environment variables, never hard-coded.

> Code comments and console messages are in Korean (original working language). See the [folder README](README.md) for the pipeline overview and the [disclaimer](../README.md#disclaimer).


In [ ]:

# NOTE: this all-in-one cell is slow on large batches.
# The split cells below run the same steps in stages - prefer those.

## ============================================================
# Amazon 상품 상세페이지 HTML 추출기
# BSR 대분류 / 하위분류 추출 개선 버전
# ============================================================

!pip -q install beautifulsoup4 lxml pandas openpyxl

import os
import re
import json
import html as html_lib
import pandas as pd

from bs4 import BeautifulSoup
from datetime import datetime
from google.colab import files
from IPython.display import display


# ============================================================
# 1. 기본 설정
# ============================================================

OUTPUT_DIR = "/content/amazon_html_extractor"
os.makedirs(OUTPUT_DIR, exist_ok=True)

COLLECTED_AT = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    f"Amazon_HTML_추출결과_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
)


# ============================================================
# 2. 결과 컬럼
# ============================================================

OUTPUT_COLUMNS = [
    "HTML파일", "ASIN", "상품URL", "브랜드", "상품명",
    "제조사", "모델번호", "UPC", "부서", "상품형태",
    "제품효능", "향", "피부타입", "사용부위", "피부톤",
    "주요특징", "활성성분", "전성분", "사용방법", "안전정보",
    "마케팅소구포인트", "제품크기", "제품용량", "수량",
    "구성품", "소재특징",

    "월간구매표시_원문", "월간구매표시_하한",
    "주간구매표시_원문", "주간구매표시_하한",
    "현재가격_USD", "고객평점", "고객리뷰수",

    # BSR
    "베스트셀러순위",
    "BSR_대분류",
    "BSR_대분류명",
    "BSR_하위순위",
    "BSR_하위분류명",

    "Amazon_스펙전체", "Amazon_원본스펙",
    "Amazon_Product_Details", "Amazon_상품특징전체",
    "Amazon_리뷰정보",

    "수집일시", "HTML복원여부", "처리상태", "오류"
]


# ============================================================
# 3. 공통 문자열 함수
# ============================================================

def clean_text(value):
    if value is None:
        return ""

    value = html_lib.unescape(str(value))
    value = value.replace("\xa0", " ")
    value = re.sub(r"[\r\n\t]+", " ", value)
    value = re.sub(r"\s+", " ", value)

    return value.strip()


def unique_text(values, separator=" / "):
    result = []
    seen = set()

    for value in values:
        value = clean_text(value)

        if not value:
            continue

        key = value.casefold()

        if key not in seen:
            result.append(value)
            seen.add(key)

    return separator.join(result)


def safe_text(element):
    if element is None:
        return ""

    return clean_text(element.get_text(" ", strip=True))


# ============================================================
# 4. HTML 읽기 및 view-source 복원
# ============================================================

def read_html_file(file_path):
    for encoding in ["utf-8", "utf-8-sig", "cp1252", "latin-1"]:
        try:
            with open(file_path, "r", encoding=encoding) as f:
                return f.read()
        except UnicodeDecodeError:
            continue

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        return f.read()


def restore_view_source_html(raw_html):
    if not raw_html:
        return "", False

    if "line-content" in raw_html:
        source_soup = BeautifulSoup(raw_html, "lxml")
        line_nodes = source_soup.select(".line-content, td.line-content")

        if line_nodes:
            restored = "\n".join(
                node.get_text("", strip=False)
                for node in line_nodes
            )

            restored = html_lib.unescape(restored)

            if re.search(
                r"<(?:html|head|body|div|table|span|script)\b",
                restored,
                re.IGNORECASE
            ):
                return restored, True

    has_html = bool(
        re.search(
            r"<(?:html|head|body|title|div|table|span|script)\b",
            raw_html,
            re.IGNORECASE
        )
    )

    if not has_html:
        decoded = html_lib.unescape(raw_html)

        if re.search(
            r"<(?:html|head|body|div|table|span|script)\b",
            decoded,
            re.IGNORECASE
        ):
            return decoded, True

    return raw_html, False


def make_soup(raw_html):
    return BeautifulSoup(raw_html, "lxml")


# ============================================================
# 5. ASIN 및 상품 URL
# ============================================================

def extract_asin(raw_html, soup, filename=""):
    patterns = [
        r"/dp/([A-Z0-9]{10})",
        r"/gp/product/([A-Z0-9]{10})",
        r'"asin"\s*:\s*"([A-Z0-9]{10})"',
        r'"ASIN"\s*:\s*"([A-Z0-9]{10})"',
        r'name=["\']ASIN["\'][^>]*value=["\']([A-Z0-9]{10})'
    ]

    for source in [raw_html, str(soup), filename]:
        for pattern in patterns:
            match = re.search(pattern, source, re.IGNORECASE)

            if match:
                return match.group(1).upper()

    return ""


def extract_product_url(raw_html, asin):
    patterns = [
        r'https?://www\.amazon\.[^"\s<>]+/dp/[A-Z0-9]{10}',
        r'https?://www\.amazon\.[^"\s<>]+/gp/product/[A-Z0-9]{10}'
    ]

    for pattern in patterns:
        match = re.search(pattern, raw_html, re.IGNORECASE)

        if match:
            return html_lib.unescape(match.group(0)).split("?")[0]

    if asin:
        return f"https://www.amazon.com/dp/{asin}"

    return ""


# ============================================================
# 6. 상품명 및 브랜드
# ============================================================

def extract_product_title(soup):
    for selector in [
        "#productTitle",
        "span#productTitle",
        "h1#title",
        'meta[property="og:title"]',
        "title"
    ]:
        element = soup.select_one(selector)

        if element is None:
            continue

        value = (
            clean_text(element.get("content", ""))
            if element.name == "meta"
            else safe_text(element)
        )

        if value:
            return re.sub(
                r"\s*:\s*Amazon\.com.*$",
                "",
                value,
                flags=re.IGNORECASE
            )

    return ""


def extract_brand(soup):
    for selector in [
        "#bylineInfo",
        "a#bylineInfo",
        "tr.po-brand td.a-span9",
        'meta[property="product:brand"]'
    ]:
        element = soup.select_one(selector)

        if element is None:
            continue

        value = (
            clean_text(element.get("content", ""))
            if element.name == "meta"
            else safe_text(element)
        )

        if value:
            value = re.sub(
                r"^(Visit the|Brand:)\s*",
                "",
                value,
                flags=re.IGNORECASE
            )
            value = re.sub(r"\s+Store$", "", value, flags=re.IGNORECASE)
            return value.strip()

    for script in soup.select('script[type="application/ld+json"]'):
        try:
            data = json.loads(script.string or script.get_text())
        except Exception:
            continue

        objects = data if isinstance(data, list) else [data]

        for obj in objects:
            if not isinstance(obj, dict):
                continue

            brand = obj.get("brand", "")

            if isinstance(brand, dict):
                brand = brand.get("name", "")

            if brand:
                return clean_text(brand)

    return ""


# ============================================================
# 7. 상품 스펙 테이블
# ============================================================

def extract_spec_tables(soup):
    specs = {}

    for table in soup.select("table"):
        for row in table.select("tr"):
            cells = row.find_all(["th", "td"], recursive=False)

            if len(cells) < 2:
                cells = row.find_all(["th", "td"])

            if len(cells) < 2:
                continue

            key = safe_text(cells[0]).rstrip(":").strip()
            value = unique_text(
                [safe_text(cell) for cell in cells[1:]],
                separator=" "
            )

            if key and value and key not in specs:
                specs[key] = value

    for item in soup.select(
        "#detailBullets_feature_div li, "
        "#detailBulletsWrapper_feature_div li"
    ):
        text = safe_text(item)

        if ":" not in text:
            continue

        key, value = text.split(":", 1)
        key = clean_text(key)
        value = clean_text(value)

        if key and value and key not in specs:
            specs[key] = value

    return specs


def format_specs(specs):
    return "\n".join(
        f"{key}: {value}"
        for key, value in specs.items()
    )


def find_section_text(soup, selectors):
    values = []

    for selector in selectors:
        for element in soup.select(selector):
            text = safe_text(element)

            if text:
                values.append(text)

    return unique_text(values, separator="\n")


def extract_product_details_text(soup):
    return find_section_text(
        soup,
        [
            "#detailBullets_feature_div",
            "#detailBulletsWrapper_feature_div",
            "#productDetails_detailBullets_sections1",
            "#productDetails_detailBullets_sections2",
            "#productDetails_techSpec_section_1",
            "#productDetails_techSpec_section_2",
            "#technicalSpecifications_section_1",
            "#technicalSpecifications_section_2",
            "#prodDetails"
        ]
    )


# ============================================================
# 8. 상품 특징
# ============================================================

def extract_feature_bullets(soup):
    values = []

    for selector in [
        "#feature-bullets li",
        "#feature-bullets .a-list-item",
        "#featurebullets_feature_div li"
    ]:
        for element in soup.select(selector):
            text = safe_text(element)

            if not text:
                continue

            if text.lower() in ["see more", "see less", "learn more"]:
                continue

            values.append(text)

    return list(dict.fromkeys(values))


# ============================================================
# 9. 성분 / 사용방법 / 안전정보
# ============================================================

def extract_ingredients(soup, specs):
    values = []

    for key, value in specs.items():
        if re.search(
            r"ingredients?|active ingredients?|inactive ingredients?",
            key,
            re.IGNORECASE
        ):
            values.append(value)

    return unique_text(values, separator=" / ")


def extract_directions(soup):
    values = []

    for element in soup.find_all(["h2", "h3", "h4", "span", "b", "strong"]):
        heading = safe_text(element)

        if not re.search(
            r"^(directions|how to use|suggested use|application|사용방법)$",
            heading,
            re.IGNORECASE
        ):
            continue

        sibling = element.find_next_sibling()

        if sibling:
            text = safe_text(sibling)

            if text:
                values.append(text)

        parent = element.parent

        if parent:
            text = safe_text(parent)

            if text and len(text) > len(heading):
                values.append(text)

    return unique_text(values, separator=" / ")


def extract_safety_info(soup):
    values = []

    for element in soup.find_all(["h2", "h3", "h4", "span", "b", "strong"]):
        heading = safe_text(element)

        if not re.search(
            r"^(safety information|warnings|warning|caution|safety|주의사항)$",
            heading,
            re.IGNORECASE
        ):
            continue

        sibling = element.find_next_sibling()

        if sibling:
            text = safe_text(sibling)

            if text:
                values.append(text)

        parent = element.parent

        if parent:
            text = safe_text(parent)

            if text and len(text) > len(heading):
                values.append(text)

    return unique_text(values, separator=" / ")


# ============================================================
# 10. 리뷰 정보
# ============================================================

def extract_customer_reviews(soup):
    rating = ""
    review_count = ""
    review_info = []

    for selector in [
        "span[data-hook='rating-out-of-text']",
        "span.a-icon-alt",
        "#acrPopover",
        "i.a-icon-star span.a-icon-alt"
    ]:
        for element in soup.select(selector):
            text = safe_text(element)

            match = re.search(
                r"(\d(?:\.\d+)?)\s*out of\s*5",
                text,
                re.IGNORECASE
            )

            if match:
                rating = match.group(1)
                break

        if rating:
            break

    for selector in [
        "#acrCustomerReviewText",
        "[data-hook='total-review-count']"
    ]:
        for element in soup.select(selector):
            text = safe_text(element)

            match = re.search(
                r"([\d,]+)\s*(?:ratings|reviews|global ratings)",
                text,
                re.IGNORECASE
            )

            if match:
                review_count = match.group(1).replace(",", "")
                break

        if review_count:
            break

    for element in soup.select("[data-hook='review']"):
        text = safe_text(element)

        if text:
            review_info.append(text)

    return {
        "고객평점": rating,
        "고객리뷰수": review_count,
        "Amazon_리뷰정보": unique_text(review_info, separator="\n")
    }


# ============================================================
# 11. 스펙 필드 매핑
# ============================================================

def get_spec_value(specs, patterns):
    for key, value in specs.items():
        for pattern in patterns:
            if re.search(pattern, key, re.IGNORECASE):
                return value

    return ""


def map_product_specs(specs):
    return {
        "제조사": get_spec_value(specs, [r"^manufacturer$"]),
        "모델번호": get_spec_value(specs, [r"item model number", r"model number"]),
        "UPC": get_spec_value(specs, [r"^upc$", r"^gtin"]),
        "부서": get_spec_value(specs, [r"^department$"]),
        "상품형태": get_spec_value(specs, [r"item form", r"product form"]),
        "제품효능": get_spec_value(specs, [r"product benefits", r"benefits"]),
        "향": get_spec_value(specs, [r"^scent$", r"scent name", r"fragrance"]),
        "피부타입": get_spec_value(specs, [r"skin type"]),
        "사용부위": get_spec_value(specs, [r"use for", r"target use body part"]),
        "피부톤": get_spec_value(specs, [r"skin tone"]),
        "주요특징": get_spec_value(specs, [r"special features", r"item highlights?", r"item highlight"]),
        "활성성분": get_spec_value(specs, [r"active ingredients?"]),
        "제품크기": get_spec_value(specs, [r"product dimensions", r"item dimensions"]),
        "제품용량": get_spec_value(specs, [r"item volume", r"net content"]),
        "수량": get_spec_value(specs, [r"number of items", r"unit count", r"item count", r"^count$"]),
        "구성품": get_spec_value(specs, [r"included components", r"components included", r"package contents"]),
        "소재특징": get_spec_value(specs, [r"material features", r"^material$", r"fabric type"])
    }


# ============================================================
# 12. 구매 표시 / BSR / 현재 가격
# ============================================================

def extract_sales_signal(raw_html, soup):
    result = {
        "월간구매표시_원문": "",
        "월간구매표시_하한": "",
        "주간구매표시_원문": "",
        "주간구매표시_하한": "",
        "현재가격_USD": "",

        "베스트셀러순위": "",
        "BSR_대분류": "",
        "BSR_대분류명": "",
        "BSR_하위순위": "",
        "BSR_하위분류명": "",

        "수집일시": COLLECTED_AT
    }

    visible_text = soup.get_text(" ", strip=True)

    # --------------------------------------------------------
    # A. 구매 표시
    # --------------------------------------------------------

    purchase_pattern = re.compile(
        r"(\d[\d,]*(?:\.\d+)?)\s*(K|M)?\s*\+\s*"
        r"bought\s+in\s+past\s+(week|month)",
        re.IGNORECASE
    )

    def parse_purchase_number(number_text, suffix):
        try:
            number = float(number_text.replace(",", ""))
            suffix = (suffix or "").upper()

            if suffix == "K":
                number *= 1_000
            elif suffix == "M":
                number *= 1_000_000

            return int(number)
        except Exception:
            return ""

    for match in purchase_pattern.finditer(visible_text):
        original = clean_text(match.group(0))
        lower_bound = parse_purchase_number(
            match.group(1),
            match.group(2)
        )

        period = match.group(3).lower()

        if period == "month" and not result["월간구매표시_원문"]:
            result["월간구매표시_원문"] = original
            result["월간구매표시_하한"] = lower_bound

        elif period == "week" and not result["주간구매표시_원문"]:
            result["주간구매표시_원문"] = original
            result["주간구매표시_하한"] = lower_bound

    # --------------------------------------------------------
    # B. BSR 추출
    #
    # 예:
    # #4 in Beauty & Personal Care
    # #1 in Facial Treatments & Masks
    #
    # BSR 원문은 가능한 한 전체를 보존하고,
    # 개별 순위와 카테고리는 별도 컬럼에 저장한다.
    # --------------------------------------------------------

        # --------------------------------------------------------
    # B. Best Sellers Rank 추출
    # --------------------------------------------------------

    bsr_candidates = []

    # 1. 스펙 테이블에서 BSR 항목 추출
    specs = extract_spec_tables(soup)

    for key, value in specs.items():
        if re.search(
            r"best sellers rank|best seller rank",
            key,
            re.IGNORECASE
        ):
            bsr_candidates.append(value)

    # 2. Amazon 상세 정보 영역에서 추출
    bsr_selectors = [
        "#detailBulletsWrapper_feature_div li",
        "#detailBullets_feature_div li",
        "#productDetails_detailBullets_sections1 tr",
        "#productDetails_detailBullets_sections2 tr",
        "#prodDetails tr",
        "#SalesRank"
    ]

    for selector in bsr_selectors:
        for element in soup.select(selector):
            text = safe_text(element)

            if re.search(
                r"best sellers rank|best seller rank",
                text,
                re.IGNORECASE
            ):
                bsr_candidates.append(text)

    # 3. 페이지 전체 텍스트에서 BSR 영역 추출
    visible_text = soup.get_text(" ", strip=True)

    bsr_context = re.search(
        r"Best Sellers Rank\s*:?\s*"
        r"(.*?)"
        r"(?=Customer Reviews|Product details|"
        r"Product Description|About this item|$)",
        visible_text,
        re.IGNORECASE
    )

    if bsr_context:
        bsr_candidates.append(bsr_context.group(1))

    # 4. 후보별로 순위 + 카테고리 추출
    rank_category_pattern = re.compile(
        r"#\s*([\d,]+)\s+in\s+"
        r"(.*?)(?="
        r"\s*\(\s*See Top"
        r"|\s*#\s*[\d,]+\s+in\s+"
        r"|\s*\|\s*Best Sellers Rank"
        r"|$)",
        re.IGNORECASE
    )

    categories = []
    seen = set()

    for candidate in bsr_candidates:
        if not candidate:
            continue

        for match in rank_category_pattern.finditer(candidate):
            rank = match.group(1).replace(",", "")
            category = clean_text(match.group(2)).strip(" ,;|")

            # BSR 뒤에 붙은 불필요한 문구 제거
            category = re.split(
                r"\b(?:ASIN|Customer Reviews|Feedback sign in|"
                r"Would you like to tell us|Product details)\b",
                category,
                maxsplit=1,
                flags=re.IGNORECASE
            )[0].strip(" ,;|")

            if not category:
                continue

            key = (rank, category.casefold())

            if key not in seen:
                categories.append({
                    "rank": rank,
                    "category": category
                })
                seen.add(key)

    # 5. 대분류와 하위분류 저장
    if categories:
        result["베스트셀러순위"] = "; ".join(
            f"#{item['rank']} in {item['category']}"
            for item in categories
        )

        result["BSR_대분류"] = categories[0]["rank"]
        result["BSR_대분류명"] = categories[0]["category"]

        if len(categories) >= 2:
            result["BSR_하위순위"] = categories[1]["rank"]
            result["BSR_하위분류명"] = categories[1]["category"]

    else:
        result["베스트셀러순위"] = ""
        result["BSR_대분류"] = ""
        result["BSR_대분류명"] = ""
        result["BSR_하위순위"] = ""
        result["BSR_하위분류명"] = ""

    # --------------------------------------------------------
    # C. 현재 가격
    # --------------------------------------------------------

    price_selectors = [
        "#corePriceDisplay_desktop_feature_div .priceToPay .a-offscreen",
        "#corePriceDisplay_desktop_feature_div .a-price:not(.a-text-price) .a-offscreen",
        "#corePrice_feature_div .priceToPay .a-offscreen",
        "#apex_desktop .priceToPay .a-offscreen",
        "#priceblock_ourprice",
        "#priceblock_dealprice",
        "#price_inside_buybox"
    ]

    for selector in price_selectors:
        elements = soup.select(selector)

        for element in elements:
            text = safe_text(element)

            match = re.search(
                r"\$\s*([\d,]+(?:\.\d{1,2})?)",
                text
            )

            if match:
                result["현재가격_USD"] = match.group(1).replace(",", "")
                break

        if result["현재가격_USD"]:
            break

    # 가격 요소를 찾지 못했을 때만 상품 가격 메타데이터 확인
    if not result["현재가격_USD"]:
        for selector in [
            'meta[property="product:price:amount"]',
            'meta[itemprop="price"]'
        ]:
            element = soup.select_one(selector)

            if element:
                value = clean_text(element.get("content", ""))

                if re.fullmatch(r"\d+(?:\.\d{1,2})?", value):
                    result["현재가격_USD"] = value
                    break

    return result


# ============================================================
# 13. HTML 파일 처리
# ============================================================

def process_html_file(file_path):
    filename = os.path.basename(file_path)

    result = {column: "" for column in OUTPUT_COLUMNS}
    result["HTML파일"] = filename
    result["수집일시"] = COLLECTED_AT
    result["처리상태"] = "실패"

    try:
        raw_html = read_html_file(file_path)

        if not raw_html.strip():
            raise ValueError("HTML 파일이 비어 있습니다.")

        restored_html, was_view_source = restore_view_source_html(raw_html)
        soup = make_soup(restored_html)

        asin = extract_asin(restored_html, soup, filename)

        result["ASIN"] = asin
        result["상품URL"] = extract_product_url(restored_html, asin)
        result["상품명"] = extract_product_title(soup)
        result["브랜드"] = extract_brand(soup)

        specs = extract_spec_tables(soup)

        result["Amazon_스펙전체"] = format_specs(specs)
        result["Amazon_원본스펙"] = format_specs(specs)
        result["Amazon_Product_Details"] = extract_product_details_text(soup)

        result.update(map_product_specs(specs))

        feature_bullets = extract_feature_bullets(soup)

        result["Amazon_상품특징전체"] = unique_text(
            feature_bullets,
            separator="\n"
        )
        result["마케팅소구포인트"] = unique_text(
            feature_bullets,
            separator=" / "
        )

        result["전성분"] = extract_ingredients(soup, specs)
        result["사용방법"] = extract_directions(soup)
        result["안전정보"] = extract_safety_info(soup)

        result.update(extract_customer_reviews(soup))
        result.update(extract_sales_signal(restored_html, soup))

        result["HTML복원여부"] = (
            "복원됨" if was_view_source else "원본사용"
        )

        if not asin:
            result["처리상태"] = "확인필요_ASIN없음"
        elif not result["상품명"]:
            result["처리상태"] = "확인필요_상품명없음"
        else:
            result["처리상태"] = "성공"

    except Exception as e:
        result["오류"] = f"{type(e).__name__}: {str(e)}"
        result["처리상태"] = "실패"

    return result


# ============================================================
# 14. HTML 파일 업로드
# ============================================================

print("=" * 80)
print("Amazon 상품 상세페이지 HTML 파일을 업로드하세요.")
print("=" * 80)

uploaded = files.upload()

html_files = []

for filename, content in uploaded.items():
    if not filename.lower().endswith((".html", ".htm")):
        print(f"건너뜀: {filename} (HTML 파일이 아님)")
        continue

    file_path = os.path.join(OUTPUT_DIR, filename)

    with open(file_path, "wb") as f:
        f.write(content)

    html_files.append(file_path)

if not html_files:
    raise ValueError("업로드된 HTML 파일이 없습니다.")

print(f"\n업로드된 HTML 파일: {len(html_files)}개")


# ============================================================
# 15. 전체 HTML 처리
# ============================================================

results = []

for i, file_path in enumerate(html_files, start=1):
    filename = os.path.basename(file_path)

    print(f"[{i}/{len(html_files)}] 처리 중: {filename}")

    result = process_html_file(file_path)
    results.append(result)

    print(
        f"  상태: {result['처리상태']} | "
        f"ASIN: {result['ASIN'] or '미확인'} | "
        f"BSR: {result['베스트셀러순위'] or '미확인'}"
    )


# ============================================================
# 16. DataFrame 생성
# ============================================================

df = pd.DataFrame(results)

for column in OUTPUT_COLUMNS:
    if column not in df.columns:
        df[column] = ""

df = df[OUTPUT_COLUMNS]


# ============================================================
# 17. 결과 확인
# ============================================================

print("\n" + "=" * 80)
print("처리 결과")
print("=" * 80)

print(f"전체 파일 수: {len(df)}")
print(f"성공: {(df['처리상태'] == '성공').sum()}")
print(f"실패: {(df['처리상태'] == '실패').sum()}")

preview_columns = [
    "HTML파일",
    "ASIN",
    "상품명",
    "베스트셀러순위",
    "BSR_대분류",
    "BSR_대분류명",
    "BSR_하위순위",
    "BSR_하위분류명",
    "처리상태",
    "오류"
]

display(df[preview_columns])


# ============================================================
# 18. Excel 저장
# ============================================================

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    df.to_excel(
        writer,
        sheet_name="제품정보",
        index=False
    )

    spec_columns = [
        "HTML파일",
        "ASIN",
        "상품URL",
        "Amazon_스펙전체",
        "Amazon_원본스펙",
        "Amazon_Product_Details",
        "Amazon_상품특징전체",
        "처리상태",
        "오류"
    ]

    df[spec_columns].to_excel(
        writer,
        sheet_name="Amazon_원본스펙",
        index=False
    )

    workbook = writer.book

    for worksheet in workbook.worksheets:
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions

        for column_cells in worksheet.columns:
            column_letter = column_cells[0].column_letter
            header = str(column_cells[0].value or "")

            if header in [
                "Amazon_스펙전체",
                "Amazon_원본스펙",
                "Amazon_Product_Details",
                "Amazon_상품특징전체",
                "Amazon_리뷰정보",
                "전성분",
                "마케팅소구포인트",
                "사용방법",
                "안전정보",
                "베스트셀러순위"
            ]:
                worksheet.column_dimensions[column_letter].width = 45
            else:
                max_length = max(
                    [len(str(cell.value or "")) for cell in column_cells[:100]]
                    or [10]
                )

                worksheet.column_dimensions[column_letter].width = min(
                    max(max_length + 2, 12),
                    35
                )

print("\nExcel 저장 완료:")
print(OUTPUT_FILE)

files.download(OUTPUT_FILE)


# Amazon HTML 추출기 셀 구성

| 셀 | 내용 | 다시 실행 |
|---|---|---|
| **CELL 1** | 라이브러리 + 기본설정 + 공통 함수 + 모든 추출 함수 | 필요할 때 |
| **CELL 2** | **HTML 업로드 + `/content/amazon_html_extractor/` 저장** | **처음 1회만** |
| **CELL 3** | 저장된 HTML 전체 처리 → DataFrame 생성 → 결과 확인 | **반복 실행 가능** |
| **CELL 4** | DataFrame → Excel 저장 + 다운로드 | **반복 실행 가능** |

## 핵심

**`files.upload()`는 CELL 2에만 둔다.**

CELL 3부터는 HTML을 다시 업로드하지 않고,  
`/content/amazon_html_extractor/`에 저장된 HTML 파일을 자동으로 불러온다.

## 실행 순서

**처음 실행할 때**

**CELL 1 → CELL 2 → CELL 3 → CELL 4**

**추출 코드를 수정해서 다시 처리할 때**

**CELL 1 → CELL 3 → CELL 4**

HTML은 **다시 업로드할 필요 없음.**

###**CELL 1 — 함수 정의까지만**

In [ ]:

# ============================================================
# Amazon 상품 상세페이지 HTML 추출기
# BSR 대분류 / 하위분류 추출 개선 버전
# ============================================================

!pip -q install beautifulsoup4 lxml pandas openpyxl

import os
import re
import json
import html as html_lib
import pandas as pd

from bs4 import BeautifulSoup
from datetime import datetime
from google.colab import files
from IPython.display import display


# ============================================================
# 1. 기본 설정
# ============================================================

OUTPUT_DIR = "/content/amazon_html_extractor"
os.makedirs(OUTPUT_DIR, exist_ok=True)

COLLECTED_AT = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    f"Amazon_HTML_추출결과_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
)


# ============================================================
# 2. 결과 컬럼
# ============================================================

OUTPUT_COLUMNS = [
    "HTML파일", "ASIN", "상품URL", "브랜드", "상품명",
    "제조사", "모델번호", "UPC", "부서", "상품형태",
    "제품효능", "향", "피부타입", "사용부위", "피부톤",
    "주요특징", "활성성분", "전성분", "사용방법", "안전정보",
    "마케팅소구포인트", "제품크기", "제품용량", "수량",
    "구성품", "소재특징",

    "월간구매표시_원문", "월간구매표시_하한",
    "주간구매표시_원문", "주간구매표시_하한",
    "현재가격_USD", "고객평점", "고객리뷰수",

    # BSR
    "베스트셀러순위",
    "BSR_대분류",
    "BSR_대분류명",
    "BSR_하위순위",
    "BSR_하위분류명",

    "Amazon_스펙전체", "Amazon_원본스펙",
    "Amazon_Product_Details", "Amazon_상품특징전체",
    "Amazon_리뷰정보",

    "수집일시", "HTML복원여부", "처리상태", "오류"
]


# ============================================================
# 3. 공통 문자열 함수
# ============================================================

def clean_text(value):
    if value is None:
        return ""

    value = html_lib.unescape(str(value))
    value = value.replace("\xa0", " ")
    value = re.sub(r"[\r\n\t]+", " ", value)
    value = re.sub(r"\s+", " ", value)

    return value.strip()


def unique_text(values, separator=" / "):
    result = []
    seen = set()

    for value in values:
        value = clean_text(value)

        if not value:
            continue

        key = value.casefold()

        if key not in seen:
            result.append(value)
            seen.add(key)

    return separator.join(result)


def safe_text(element):
    if element is None:
        return ""

    return clean_text(element.get_text(" ", strip=True))


# ============================================================
# 4. HTML 읽기 및 view-source 복원
# ============================================================

def read_html_file(file_path):
    for encoding in ["utf-8", "utf-8-sig", "cp1252", "latin-1"]:
        try:
            with open(file_path, "r", encoding=encoding) as f:
                return f.read()
        except UnicodeDecodeError:
            continue

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        return f.read()


def restore_view_source_html(raw_html):
    if not raw_html:
        return "", False

    if "line-content" in raw_html:
        source_soup = BeautifulSoup(raw_html, "lxml")
        line_nodes = source_soup.select(".line-content, td.line-content")

        if line_nodes:
            restored = "\n".join(
                node.get_text("", strip=False)
                for node in line_nodes
            )

            restored = html_lib.unescape(restored)

            if re.search(
                r"<(?:html|head|body|div|table|span|script)\b",
                restored,
                re.IGNORECASE
            ):
                return restored, True

    has_html = bool(
        re.search(
            r"<(?:html|head|body|title|div|table|span|script)\b",
            raw_html,
            re.IGNORECASE
        )
    )

    if not has_html:
        decoded = html_lib.unescape(raw_html)

        if re.search(
            r"<(?:html|head|body|div|table|span|script)\b",
            decoded,
            re.IGNORECASE
        ):
            return decoded, True

    return raw_html, False


def make_soup(raw_html):
    return BeautifulSoup(raw_html, "lxml")


# ============================================================
# 5. ASIN 및 상품 URL
# ============================================================

def extract_asin(raw_html, soup, filename=""):
    patterns = [
        r"/dp/([A-Z0-9]{10})",
        r"/gp/product/([A-Z0-9]{10})",
        r'"asin"\s*:\s*"([A-Z0-9]{10})"',
        r'"ASIN"\s*:\s*"([A-Z0-9]{10})"',
        r'name=["\']ASIN["\'][^>]*value=["\']([A-Z0-9]{10})'
    ]

    for source in [raw_html, str(soup), filename]:
        for pattern in patterns:
            match = re.search(pattern, source, re.IGNORECASE)

            if match:
                return match.group(1).upper()

    return ""


def extract_product_url(raw_html, asin):
    patterns = [
        r'https?://www\.amazon\.[^"\s<>]+/dp/[A-Z0-9]{10}',
        r'https?://www\.amazon\.[^"\s<>]+/gp/product/[A-Z0-9]{10}'
    ]

    for pattern in patterns:
        match = re.search(pattern, raw_html, re.IGNORECASE)

        if match:
            return html_lib.unescape(match.group(0)).split("?")[0]

    if asin:
        return f"https://www.amazon.com/dp/{asin}"

    return ""


# ============================================================
# 6. 상품명 및 브랜드
# ============================================================

def extract_product_title(soup):
    for selector in [
        "#productTitle",
        "span#productTitle",
        "h1#title",
        'meta[property="og:title"]',
        "title"
    ]:
        element = soup.select_one(selector)

        if element is None:
            continue

        value = (
            clean_text(element.get("content", ""))
            if element.name == "meta"
            else safe_text(element)
        )

        if value:
            return re.sub(
                r"\s*:\s*Amazon\.com.*$",
                "",
                value,
                flags=re.IGNORECASE
            )

    return ""


def extract_brand(soup):
    for selector in [
        "#bylineInfo",
        "a#bylineInfo",
        "tr.po-brand td.a-span9",
        'meta[property="product:brand"]'
    ]:
        element = soup.select_one(selector)

        if element is None:
            continue

        value = (
            clean_text(element.get("content", ""))
            if element.name == "meta"
            else safe_text(element)
        )

        if value:
            value = re.sub(
                r"^(Visit the|Brand:)\s*",
                "",
                value,
                flags=re.IGNORECASE
            )
            value = re.sub(r"\s+Store$", "", value, flags=re.IGNORECASE)
            return value.strip()

    for script in soup.select('script[type="application/ld+json"]'):
        try:
            data = json.loads(script.string or script.get_text())
        except Exception:
            continue

        objects = data if isinstance(data, list) else [data]

        for obj in objects:
            if not isinstance(obj, dict):
                continue

            brand = obj.get("brand", "")

            if isinstance(brand, dict):
                brand = brand.get("name", "")

            if brand:
                return clean_text(brand)

    return ""


# ============================================================
# 7. 상품 스펙 테이블
# ============================================================

def extract_spec_tables(soup):
    specs = {}

    for table in soup.select("table"):
        for row in table.select("tr"):
            cells = row.find_all(["th", "td"], recursive=False)

            if len(cells) < 2:
                cells = row.find_all(["th", "td"])

            if len(cells) < 2:
                continue

            key = safe_text(cells[0]).rstrip(":").strip()
            value = unique_text(
                [safe_text(cell) for cell in cells[1:]],
                separator=" "
            )

            if key and value and key not in specs:
                specs[key] = value

    for item in soup.select(
        "#detailBullets_feature_div li, "
        "#detailBulletsWrapper_feature_div li"
    ):
        text = safe_text(item)

        if ":" not in text:
            continue

        key, value = text.split(":", 1)
        key = clean_text(key)
        value = clean_text(value)

        if key and value and key not in specs:
            specs[key] = value

    return specs


def format_specs(specs):
    return "\n".join(
        f"{key}: {value}"
        for key, value in specs.items()
    )


def find_section_text(soup, selectors):
    values = []

    for selector in selectors:
        for element in soup.select(selector):
            text = safe_text(element)

            if text:
                values.append(text)

    return unique_text(values, separator="\n")


def extract_product_details_text(soup):
    return find_section_text(
        soup,
        [
            "#detailBullets_feature_div",
            "#detailBulletsWrapper_feature_div",
            "#productDetails_detailBullets_sections1",
            "#productDetails_detailBullets_sections2",
            "#productDetails_techSpec_section_1",
            "#productDetails_techSpec_section_2",
            "#technicalSpecifications_section_1",
            "#technicalSpecifications_section_2",
            "#prodDetails"
        ]
    )


# ============================================================
# 8. 상품 특징
# ============================================================

def extract_feature_bullets(soup):
    values = []

    for selector in [
        "#feature-bullets li",
        "#feature-bullets .a-list-item",
        "#featurebullets_feature_div li"
    ]:
        for element in soup.select(selector):
            text = safe_text(element)

            if not text:
                continue

            if text.lower() in ["see more", "see less", "learn more"]:
                continue

            values.append(text)

    return list(dict.fromkeys(values))


# ============================================================
# 9. 성분 / 사용방법 / 안전정보
# ============================================================

def extract_ingredients(soup, specs):
    values = []

    for key, value in specs.items():
        if re.search(
            r"ingredients?|active ingredients?|inactive ingredients?",
            key,
            re.IGNORECASE
        ):
            values.append(value)

    return unique_text(values, separator=" / ")


def extract_directions(soup):
    values = []

    for element in soup.find_all(["h2", "h3", "h4", "span", "b", "strong"]):
        heading = safe_text(element)

        if not re.search(
            r"^(directions|how to use|suggested use|application|사용방법)$",
            heading,
            re.IGNORECASE
        ):
            continue

        sibling = element.find_next_sibling()

        if sibling:
            text = safe_text(sibling)

            if text:
                values.append(text)

        parent = element.parent

        if parent:
            text = safe_text(parent)

            if text and len(text) > len(heading):
                values.append(text)

    return unique_text(values, separator=" / ")


def extract_safety_info(soup):
    values = []

    for element in soup.find_all(["h2", "h3", "h4", "span", "b", "strong"]):
        heading = safe_text(element)

        if not re.search(
            r"^(safety information|warnings|warning|caution|safety|주의사항)$",
            heading,
            re.IGNORECASE
        ):
            continue

        sibling = element.find_next_sibling()

        if sibling:
            text = safe_text(sibling)

            if text:
                values.append(text)

        parent = element.parent

        if parent:
            text = safe_text(parent)

            if text and len(text) > len(heading):
                values.append(text)

    return unique_text(values, separator=" / ")


# ============================================================
# 10. 리뷰 정보
# ============================================================

def extract_customer_reviews(soup):
    rating = ""
    review_count = ""
    review_info = []

    for selector in [
        "span[data-hook='rating-out-of-text']",
        "span.a-icon-alt",
        "#acrPopover",
        "i.a-icon-star span.a-icon-alt"
    ]:
        for element in soup.select(selector):
            text = safe_text(element)

            match = re.search(
                r"(\d(?:\.\d+)?)\s*out of\s*5",
                text,
                re.IGNORECASE
            )

            if match:
                rating = match.group(1)
                break

        if rating:
            break

    for selector in [
        "#acrCustomerReviewText",
        "[data-hook='total-review-count']"
    ]:
        for element in soup.select(selector):
            text = safe_text(element)

            match = re.search(
                r"([\d,]+)\s*(?:ratings|reviews|global ratings)",
                text,
                re.IGNORECASE
            )

            if match:
                review_count = match.group(1).replace(",", "")
                break

        if review_count:
            break

    for element in soup.select("[data-hook='review']"):
        text = safe_text(element)

        if text:
            review_info.append(text)

    return {
        "고객평점": rating,
        "고객리뷰수": review_count,
        "Amazon_리뷰정보": unique_text(review_info, separator="\n")
    }


# ============================================================
# 11. 스펙 필드 매핑
# ============================================================

def get_spec_value(specs, patterns):
    for key, value in specs.items():
        for pattern in patterns:
            if re.search(pattern, key, re.IGNORECASE):
                return value

    return ""


def map_product_specs(specs):
    return {
        "제조사": get_spec_value(specs, [r"^manufacturer$"]),
        "모델번호": get_spec_value(specs, [r"item model number", r"model number"]),
        "UPC": get_spec_value(specs, [r"^upc$", r"^gtin"]),
        "부서": get_spec_value(specs, [r"^department$"]),
        "상품형태": get_spec_value(specs, [r"item form", r"product form"]),
        "제품효능": get_spec_value(specs, [r"product benefits", r"benefits"]),
        "향": get_spec_value(specs, [r"^scent$", r"scent name", r"fragrance"]),
        "피부타입": get_spec_value(specs, [r"skin type"]),
        "사용부위": get_spec_value(specs, [r"use for", r"target use body part"]),
        "피부톤": get_spec_value(specs, [r"skin tone"]),
        "주요특징": get_spec_value(specs, [r"special features", r"item highlights?", r"item highlight"]),
        "활성성분": get_spec_value(specs, [r"active ingredients?"]),
        "제품크기": get_spec_value(specs, [r"product dimensions", r"item dimensions"]),
        "제품용량": get_spec_value(specs, [r"item volume", r"net content"]),
        "수량": get_spec_value(specs, [r"number of items", r"unit count", r"item count", r"^count$"]),
        "구성품": get_spec_value(specs, [r"included components", r"components included", r"package contents"]),
        "소재특징": get_spec_value(specs, [r"material features", r"^material$", r"fabric type"])
    }


# ============================================================
# 12. 구매 표시 / BSR / 현재 가격
# ============================================================

def extract_sales_signal(raw_html, soup):
    result = {
        "월간구매표시_원문": "",
        "월간구매표시_하한": "",
        "주간구매표시_원문": "",
        "주간구매표시_하한": "",
        "현재가격_USD": "",

        "베스트셀러순위": "",
        "BSR_대분류": "",
        "BSR_대분류명": "",
        "BSR_하위순위": "",
        "BSR_하위분류명": "",

        "수집일시": COLLECTED_AT
    }

    visible_text = soup.get_text(" ", strip=True)

    # --------------------------------------------------------
    # A. 구매 표시
    # --------------------------------------------------------

    purchase_pattern = re.compile(
        r"(\d[\d,]*(?:\.\d+)?)\s*(K|M)?\s*\+\s*"
        r"bought\s+in\s+past\s+(week|month)",
        re.IGNORECASE
    )

    def parse_purchase_number(number_text, suffix):
        try:
            number = float(number_text.replace(",", ""))
            suffix = (suffix or "").upper()

            if suffix == "K":
                number *= 1_000
            elif suffix == "M":
                number *= 1_000_000

            return int(number)
        except Exception:
            return ""

    for match in purchase_pattern.finditer(visible_text):
        original = clean_text(match.group(0))
        lower_bound = parse_purchase_number(
            match.group(1),
            match.group(2)
        )

        period = match.group(3).lower()

        if period == "month" and not result["월간구매표시_원문"]:
            result["월간구매표시_원문"] = original
            result["월간구매표시_하한"] = lower_bound

        elif period == "week" and not result["주간구매표시_원문"]:
            result["주간구매표시_원문"] = original
            result["주간구매표시_하한"] = lower_bound

    # --------------------------------------------------------
    # B. BSR 추출
    #
    # 예:
    # #4 in Beauty & Personal Care
    # #1 in Facial Treatments & Masks
    #
    # BSR 원문은 가능한 한 전체를 보존하고,
    # 개별 순위와 카테고리는 별도 컬럼에 저장한다.
    # --------------------------------------------------------

        # --------------------------------------------------------
    # B. Best Sellers Rank 추출
    # --------------------------------------------------------

    bsr_candidates = []

    # 1. 스펙 테이블에서 BSR 항목 추출
    specs = extract_spec_tables(soup)

    for key, value in specs.items():
        if re.search(
            r"best sellers rank|best seller rank",
            key,
            re.IGNORECASE
        ):
            bsr_candidates.append(value)

    # 2. Amazon 상세 정보 영역에서 추출
    bsr_selectors = [
        "#detailBulletsWrapper_feature_div li",
        "#detailBullets_feature_div li",
        "#productDetails_detailBullets_sections1 tr",
        "#productDetails_detailBullets_sections2 tr",
        "#prodDetails tr",
        "#SalesRank"
    ]

    for selector in bsr_selectors:
        for element in soup.select(selector):
            text = safe_text(element)

            if re.search(
                r"best sellers rank|best seller rank",
                text,
                re.IGNORECASE
            ):
                bsr_candidates.append(text)

    # 3. 페이지 전체 텍스트에서 BSR 영역 추출
    visible_text = soup.get_text(" ", strip=True)

    bsr_context = re.search(
        r"Best Sellers Rank\s*:?\s*"
        r"(.*?)"
        r"(?=Customer Reviews|Product details|"
        r"Product Description|About this item|$)",
        visible_text,
        re.IGNORECASE
    )

    if bsr_context:
        bsr_candidates.append(bsr_context.group(1))

    # 4. 후보별로 순위 + 카테고리 추출
    rank_category_pattern = re.compile(
        r"#\s*([\d,]+)\s+in\s+"
        r"(.*?)(?="
        r"\s*\(\s*See Top"
        r"|\s*#\s*[\d,]+\s+in\s+"
        r"|\s*\|\s*Best Sellers Rank"
        r"|$)",
        re.IGNORECASE
    )

    categories = []
    seen = set()

    for candidate in bsr_candidates:
        if not candidate:
            continue

        for match in rank_category_pattern.finditer(candidate):
            rank = match.group(1).replace(",", "")
            category = clean_text(match.group(2)).strip(" ,;|")

            # BSR 뒤에 붙은 불필요한 문구 제거
            category = re.split(
                r"\b(?:ASIN|Customer Reviews|Feedback sign in|"
                r"Would you like to tell us|Product details)\b",
                category,
                maxsplit=1,
                flags=re.IGNORECASE
            )[0].strip(" ,;|")

            if not category:
                continue

            key = (rank, category.casefold())

            if key not in seen:
                categories.append({
                    "rank": rank,
                    "category": category
                })
                seen.add(key)

    # 5. 대분류와 하위분류 저장
    if categories:
        result["베스트셀러순위"] = "; ".join(
            f"#{item['rank']} in {item['category']}"
            for item in categories
        )

        result["BSR_대분류"] = categories[0]["rank"]
        result["BSR_대분류명"] = categories[0]["category"]

        if len(categories) >= 2:
            result["BSR_하위순위"] = categories[1]["rank"]
            result["BSR_하위분류명"] = categories[1]["category"]

    else:
        result["베스트셀러순위"] = ""
        result["BSR_대분류"] = ""
        result["BSR_대분류명"] = ""
        result["BSR_하위순위"] = ""
        result["BSR_하위분류명"] = ""

    # --------------------------------------------------------
    # C. 현재 가격
    # --------------------------------------------------------

    price_selectors = [
        "#corePriceDisplay_desktop_feature_div .priceToPay .a-offscreen",
        "#corePriceDisplay_desktop_feature_div .a-price:not(.a-text-price) .a-offscreen",
        "#corePrice_feature_div .priceToPay .a-offscreen",
        "#apex_desktop .priceToPay .a-offscreen",
        "#priceblock_ourprice",
        "#priceblock_dealprice",
        "#price_inside_buybox"
    ]

    for selector in price_selectors:
        elements = soup.select(selector)

        for element in elements:
            text = safe_text(element)

            match = re.search(
                r"\$\s*([\d,]+(?:\.\d{1,2})?)",
                text
            )

            if match:
                result["현재가격_USD"] = match.group(1).replace(",", "")
                break

        if result["현재가격_USD"]:
            break

    # 가격 요소를 찾지 못했을 때만 상품 가격 메타데이터 확인
    if not result["현재가격_USD"]:
        for selector in [
            'meta[property="product:price:amount"]',
            'meta[itemprop="price"]'
        ]:
            element = soup.select_one(selector)

            if element:
                value = clean_text(element.get("content", ""))

                if re.fullmatch(r"\d+(?:\.\d{1,2})?", value):
                    result["현재가격_USD"] = value
                    break

    return result


# ============================================================
# 13. HTML 파일 처리
# ============================================================

def process_html_file(file_path):
    filename = os.path.basename(file_path)

    result = {column: "" for column in OUTPUT_COLUMNS}
    result["HTML파일"] = filename
    result["수집일시"] = COLLECTED_AT
    result["처리상태"] = "실패"

    try:
        raw_html = read_html_file(file_path)

        if not raw_html.strip():
            raise ValueError("HTML 파일이 비어 있습니다.")

        restored_html, was_view_source = restore_view_source_html(raw_html)
        soup = make_soup(restored_html)

        asin = extract_asin(restored_html, soup, filename)

        result["ASIN"] = asin
        result["상품URL"] = extract_product_url(restored_html, asin)
        result["상품명"] = extract_product_title(soup)
        result["브랜드"] = extract_brand(soup)

        specs = extract_spec_tables(soup)

        result["Amazon_스펙전체"] = format_specs(specs)
        result["Amazon_원본스펙"] = format_specs(specs)
        result["Amazon_Product_Details"] = extract_product_details_text(soup)

        result.update(map_product_specs(specs))

        feature_bullets = extract_feature_bullets(soup)

        result["Amazon_상품특징전체"] = unique_text(
            feature_bullets,
            separator="\n"
        )
        result["마케팅소구포인트"] = unique_text(
            feature_bullets,
            separator=" / "
        )

        result["전성분"] = extract_ingredients(soup, specs)
        result["사용방법"] = extract_directions(soup)
        result["안전정보"] = extract_safety_info(soup)

        result.update(extract_customer_reviews(soup))
        result.update(extract_sales_signal(restored_html, soup))

        result["HTML복원여부"] = (
            "복원됨" if was_view_source else "원본사용"
        )

        if not asin:
            result["처리상태"] = "확인필요_ASIN없음"
        elif not result["상품명"]:
            result["처리상태"] = "확인필요_상품명없음"
        else:
            result["처리상태"] = "성공"

    except Exception as e:
        result["오류"] = f"{type(e).__name__}: {str(e)}"
        result["처리상태"] = "실패"

    return result




###**CELL 2 — HTML 업로드 전용 ⭐ 한 번만 실행**

In [ ]:
# ============================================================
# [CELL 2] Amazon HTML 파일 업로드
# ※ 이 셀은 처음 한 번만 실행
# ===========================================
# =================

print("=" * 80)
print("Amazon 상품 상세페이지 HTML 파일을 업로드하세요.")
print("=" * 80)

uploaded = files.upload()

html_files = []

for filename, content in uploaded.items():

    if not filename.lower().endswith((".html", ".htm")):
        print(f"건너뜀: {filename} (HTML 파일이 아님)")
        continue

    file_path = os.path.join(OUTPUT_DIR, filename)

    with open(file_path, "wb") as f:
        f.write(content)

    html_files.append(file_path)

if not html_files:
    raise ValueError("업로드된 HTML 파일이 없습니다.")

print(f"\n업로드된 HTML 파일: {len(html_files)}개")

for file_path in html_files:
    print(" -", os.path.basename(file_path))

###**CELL 3 — 업로드 없이 저장된 HTML 다시 처리**

In [ ]:
# ============================================================
# [CELL 3] 저장된 HTML 전체 처리
# ※ HTML 재업로드 없음
# ============================================================

# ------------------------------------------------------------
# 1. 기존 저장 폴더에서 HTML 자동 검색
# ------------------------------------------------------------

html_files = []

for filename in os.listdir(OUTPUT_DIR):

    if filename.lower().endswith((".html", ".htm")):

        file_path = os.path.join(
            OUTPUT_DIR,
            filename
        )

        if os.path.isfile(file_path):
            html_files.append(file_path)


html_files = sorted(html_files)


if not html_files:
    raise ValueError(
        "저장된 HTML 파일이 없습니다.\n"
        "먼저 CELL 2에서 HTML 파일을 업로드하세요."
    )


print("=" * 80)
print("저장된 HTML 파일 처리")
print("=" * 80)

print(f"처리할 HTML 파일: {len(html_files)}개")


# ------------------------------------------------------------
# 2. 전체 HTML 처리
# ------------------------------------------------------------

results = []

for i, file_path in enumerate(html_files, start=1):

    filename = os.path.basename(file_path)

    print(
        f"[{i}/{len(html_files)}] 처리 중: {filename}"
    )

    result = process_html_file(file_path)

    results.append(result)

    print(
        f"  상태: {result['처리상태']} | "
        f"ASIN: {result['ASIN'] or '미확인'} | "
        f"BSR: {result['베스트셀러순위'] or '미확인'}"
    )


# ------------------------------------------------------------
# 3. DataFrame 생성
# ------------------------------------------------------------

df = pd.DataFrame(results)

for column in OUTPUT_COLUMNS:

    if column not in df.columns:
        df[column] = ""


df = df[OUTPUT_COLUMNS]


# ------------------------------------------------------------
# 4. 결과 확인
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("처리 결과")
print("=" * 80)

print(f"전체 파일 수: {len(df)}")
print(
    f"성공: {(df['처리상태'] == '성공').sum()}"
)
print(
    f"실패: {(df['처리상태'] == '실패').sum()}"
)


preview_columns = [
    "HTML파일",
    "ASIN",
    "상품명",
    "베스트셀러순위",
    "BSR_대분류",
    "BSR_대분류명",
    "BSR_하위순위",
    "BSR_하위분류명",
    "처리상태",
    "오류"
]

display(df[preview_columns])

###**CELL 4 — Excel 저장**

In [ ]:
# ============================================================
# [CELL 4] Excel 저장 및 다운로드
# ============================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    # --------------------------------------------------------
    # 제품정보
    # --------------------------------------------------------

    df.to_excel(
        writer,
        sheet_name="제품정보",
        index=False
    )

    # --------------------------------------------------------
    # Amazon 원본 스펙
    # --------------------------------------------------------

    spec_columns = [
        "HTML파일",
        "ASIN",
        "상품URL",
        "Amazon_스펙전체",
        "Amazon_원본스펙",
        "Amazon_Product_Details",
        "Amazon_상품특징전체",
        "처리상태",
        "오류"
    ]

    df[spec_columns].to_excel(
        writer,
        sheet_name="Amazon_원본스펙",
        index=False
    )

    # --------------------------------------------------------
    # Excel 서식
    # --------------------------------------------------------

    workbook = writer.book

    for worksheet in workbook.worksheets:

        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions

        for column_cells in worksheet.columns:

            column_letter = column_cells[0].column_letter
            header = str(
                column_cells[0].value or ""
            )

            if header in [
                "Amazon_스펙전체",
                "Amazon_원본스펙",
                "Amazon_Product_Details",
                "Amazon_상품특징전체",
                "Amazon_리뷰정보",
                "전성분",
                "마케팅소구포인트",
                "사용방법",
                "안전정보",
                "베스트셀러순위"
            ]:

                worksheet.column_dimensions[
                    column_letter
                ].width = 45

            else:

                max_length = max(
                    [
                        len(str(cell.value or ""))
                        for cell in column_cells[:100]
                    ]
                    or [10]
                )

                worksheet.column_dimensions[
                    column_letter
                ].width = min(
                    max(max_length + 2, 12),
                    35
                )


print("\n" + "=" * 80)
print("Excel 저장 완료")
print("=" * 80)

print(OUTPUT_FILE)

files.download(OUTPUT_FILE)